<a href="https://colab.research.google.com/github/edakilic/BMI-Calculator/blob/master/production-scheduling-project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# FINAL PROJECT CODE
# =========================================================

import random
import time
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================================================
# PARAMETERS
# =========================================================

POPULATION_SIZE = 100
GENERATIONS = 300
MUTATION_RATE = 0.10
TOURNAMENT_SIZE = 5
LOCAL_SEARCH_ITER = 5
NUM_RUNS = 10

# =========================================================
# BENCHMARK GENERATION
# =========================================================

def generate_benchmark(num_jobs, num_machines, seed):
    np.random.seed(seed)
    return np.random.randint(1, 100, size=(num_jobs, num_machines))


benchmarks = [
    {
        "name": "Easy_20x5",
        "difficulty": "Easy",
        "jobs": 20,
        "machines": 5,
        "seed": 42
    },
    {
        "name": "Moderate_30x5",
        "difficulty": "Moderate",
        "jobs": 30,
        "machines": 5,
        "seed": 43
    },
    {
        "name": "Difficult_50x10",
        "difficulty": "Difficult",
        "jobs": 50,
        "machines": 10,
        "seed": 44
    }
]


methods = [
    {
        "name": "Baseline GA",
        "use_local_search": False
    },
    {
        "name": "Hybrid GA",
        "use_local_search": True
    }
]

# =========================================================
# MAKESPAN CALCULATION
# =========================================================

def calculate_makespan(sequence, processing_times):
    n_jobs = len(sequence)
    n_machines = processing_times.shape[1]

    completion = np.zeros((n_jobs, n_machines))

    for i in range(n_jobs):
        job = sequence[i]

        for m in range(n_machines):
            if i == 0 and m == 0:
                completion[i][m] = processing_times[job][m]

            elif i == 0:
                completion[i][m] = completion[i][m - 1] + processing_times[job][m]

            elif m == 0:
                completion[i][m] = completion[i - 1][m] + processing_times[job][m]

            else:
                completion[i][m] = max(
                    completion[i - 1][m],
                    completion[i][m - 1]
                ) + processing_times[job][m]

    return completion[-1][-1]

# =========================================================
# INITIAL POPULATION
# =========================================================

def create_population(num_jobs):
    population = []

    for _ in range(POPULATION_SIZE):
        chromosome = list(range(num_jobs))
        random.shuffle(chromosome)
        population.append(chromosome)

    return population

# =========================================================
# SELECTION - TOURNAMENT SELECTION
# =========================================================

def selection(population, processing_times):
    tournament = random.sample(population, TOURNAMENT_SIZE)

    tournament = sorted(
        tournament,
        key=lambda x: calculate_makespan(x, processing_times)
    )

    return tournament[0]

# =========================================================
# CROSSOVER - ORDER CROSSOVER
# =========================================================

def crossover(parent1, parent2):
    size = len(parent1)

    start, end = sorted(random.sample(range(size), 2))

    child = [-1] * size
    child[start:end] = parent1[start:end]

    ptr = 0

    for gene in parent2:
        if gene not in child:
            while child[ptr] != -1:
                ptr += 1
            child[ptr] = gene

    return child

# =========================================================
# MUTATION - SWAP MUTATION
# =========================================================

def mutation(chromosome):
    chromosome = chromosome.copy()

    if random.random() < MUTATION_RATE:
        i, j = random.sample(range(len(chromosome)), 2)
        chromosome[i], chromosome[j] = chromosome[j], chromosome[i]

    return chromosome

# =========================================================
# LOCAL SEARCH - SWAP-BASED LOCAL SEARCH
# =========================================================

def local_search(chromosome, processing_times):
    best = chromosome.copy()
    best_makespan = calculate_makespan(best, processing_times)

    for _ in range(LOCAL_SEARCH_ITER):
        neighbor = best.copy()

        i, j = random.sample(range(len(neighbor)), 2)
        neighbor[i], neighbor[j] = neighbor[j], neighbor[i]

        neighbor_makespan = calculate_makespan(neighbor, processing_times)

        if neighbor_makespan < best_makespan:
            best = neighbor
            best_makespan = neighbor_makespan

    return best

# =========================================================
# GA RUN FUNCTION
# =========================================================

def run_ga(run_id, processing_times, use_local_search):
    start_time = time.time()

    num_jobs = processing_times.shape[0]

    population = create_population(num_jobs)

    best_solution = None
    best_makespan = float("inf")
    history = []

    for generation in range(GENERATIONS):
        new_population = []

        for _ in range(POPULATION_SIZE):
            parent1 = selection(population, processing_times)
            parent2 = selection(population, processing_times)

            child = crossover(parent1, parent2)
            child = mutation(child)

            if use_local_search:
                child = local_search(child, processing_times)

            new_population.append(child)

        population = new_population

        generation_best = min(
            population,
            key=lambda x: calculate_makespan(x, processing_times)
        )

        generation_makespan = calculate_makespan(
            generation_best,
            processing_times
        )

        if generation_makespan < best_makespan:
            best_makespan = generation_makespan
            best_solution = generation_best

        history.append(best_makespan)

    runtime = time.time() - start_time

    return {
        "Run": run_id,
        "Best_Makespan": best_makespan,
        "Runtime_seconds": runtime,
        "Best_Sequence": best_solution,
        "History": history
    }

# =========================================================
# GANTT CHART FUNCTION
# =========================================================

def gantt_chart(sequence, processing_times, filename, title):
    n_jobs = len(sequence)
    n_machines = processing_times.shape[1]

    completion = np.zeros((n_jobs, n_machines))
    start_times = np.zeros((n_jobs, n_machines))

    for i in range(n_jobs):
        job = sequence[i]

        for m in range(n_machines):
            if i == 0 and m == 0:
                start_times[i][m] = 0

            elif i == 0:
                start_times[i][m] = completion[i][m - 1]

            elif m == 0:
                start_times[i][m] = completion[i - 1][m]

            else:
                start_times[i][m] = max(
                    completion[i - 1][m],
                    completion[i][m - 1]
                )

            completion[i][m] = start_times[i][m] + processing_times[job][m]

    plt.figure(figsize=(14, 7))

    for i in range(n_jobs):
        job = sequence[i]

        for m in range(n_machines):
            plt.barh(
                y=m,
                width=processing_times[job][m],
                left=start_times[i][m]
            )

            if n_jobs <= 30:
                plt.text(
                    start_times[i][m] + processing_times[job][m] / 2,
                    m,
                    f"J{job + 1}",
                    ha="center",
                    va="center",
                    fontsize=7
                )

    plt.xlabel("Time")
    plt.ylabel("Machine")
    plt.title(title)
    plt.yticks(
        range(n_machines),
        [f"M{m + 1}" for m in range(n_machines)]
    )
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(filename)
    plt.show()

# =========================================================
# MAIN EXPERIMENT
# =========================================================

os.makedirs("results", exist_ok=True)

all_summaries = []
all_run_details = []

best_hybrid_overall = None

for benchmark in benchmarks:
    print("\n===================================================")
    print(f"Benchmark: {benchmark['name']}")
    print(f"Difficulty: {benchmark['difficulty']}")
    print(f"Size: {benchmark['jobs']} jobs x {benchmark['machines']} machines")
    print("===================================================")

    processing_times = generate_benchmark(
        benchmark["jobs"],
        benchmark["machines"],
        benchmark["seed"]
    )

    for method in methods:
        print(f"\nMethod: {method['name']}")

        method_results = []

        for run in range(1, NUM_RUNS + 1):
            print(f"Run {run} started...")

            result = run_ga(
                run_id=run,
                processing_times=processing_times,
                use_local_search=method["use_local_search"]
            )

            method_results.append(result)

            all_run_details.append({
                "Benchmark": benchmark["name"],
                "Difficulty": benchmark["difficulty"],
                "Jobs": benchmark["jobs"],
                "Machines": benchmark["machines"],
                "Method": method["name"],
                "Run": result["Run"],
                "Best_Makespan": result["Best_Makespan"],
                "Runtime_seconds": result["Runtime_seconds"],
                "Best_Sequence": result["Best_Sequence"]
            })

            print(
                f"Run {run} finished | "
                f"Best Makespan: {result['Best_Makespan']} | "
                f"Runtime: {result['Runtime_seconds']:.2f} sec"
            )

        makespans = [
            r["Best_Makespan"]
            for r in method_results
        ]

        runtimes = [
            r["Runtime_seconds"]
            for r in method_results
        ]

        best_run = min(
            method_results,
            key=lambda x: x["Best_Makespan"]
        )

        summary = {
            "Benchmark": benchmark["name"],
            "Difficulty": benchmark["difficulty"],
            "Jobs": benchmark["jobs"],
            "Machines": benchmark["machines"],
            "Method": method["name"],
            "Mean_Makespan": np.mean(makespans),
            "Std_Dev": np.std(makespans),
            "Best_Makespan": np.min(makespans),
            "Worst_Makespan": np.max(makespans),
            "Average_Runtime_seconds": np.mean(runtimes)
        }

        all_summaries.append(summary)

        safe_method_name = method["name"].replace(" ", "_")

        plt.figure(figsize=(10, 5))
        plt.plot(best_run["History"])
        plt.xlabel("Generation")
        plt.ylabel("Best Makespan")
        plt.title(
            f"Convergence Curve - {benchmark['name']} - {method['name']}"
        )
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(
            f"results/convergence_{benchmark['name']}_{safe_method_name}.png"
        )
        plt.show()

        if method["name"] == "Hybrid GA":
            if (
                best_hybrid_overall is None
                or best_run["Best_Makespan"] < best_hybrid_overall["Best_Makespan"]
            ):
                best_hybrid_overall = {
                    "Benchmark": benchmark["name"],
                    "Processing_Times": processing_times,
                    "Best_Sequence": best_run["Best_Sequence"],
                    "Best_Makespan": best_run["Best_Makespan"],
                    "History": best_run["History"]
                }

# =========================================================
# SAVE RESULTS
# =========================================================

summary_df = pd.DataFrame(all_summaries)
details_df = pd.DataFrame(all_run_details)

summary_df.to_csv(
    "results/method_comparison_summary.csv",
    index=False
)

details_df.to_csv(
    "results/all_run_details.csv",
    index=False
)

# =========================================================
# BASELINE GA vs HYBRID GA IMPROVEMENT TABLE
# =========================================================

comparison_rows = []

for benchmark in benchmarks:
    bname = benchmark["name"]

    base_row = summary_df[
        (summary_df["Benchmark"] == bname)
        & (summary_df["Method"] == "Baseline GA")
    ].iloc[0]

    hybrid_row = summary_df[
        (summary_df["Benchmark"] == bname)
        & (summary_df["Method"] == "Hybrid GA")
    ].iloc[0]

    improvement_percent = (
        (base_row["Best_Makespan"] - hybrid_row["Best_Makespan"])
        / base_row["Best_Makespan"]
    ) * 100

    comparison_rows.append({
        "Benchmark": bname,
        "Baseline_Best": base_row["Best_Makespan"],
        "Hybrid_Best": hybrid_row["Best_Makespan"],
        "Improvement_Percent": improvement_percent,
        "Baseline_Avg_Runtime": base_row["Average_Runtime_seconds"],
        "Hybrid_Avg_Runtime": hybrid_row["Average_Runtime_seconds"]
    })

comparison_df = pd.DataFrame(comparison_rows)

comparison_df.to_csv(
    "results/baseline_vs_hybrid_comparison.csv",
    index=False
)

# =========================================================
# GANTT CHART FOR BEST HYBRID SOLUTION
# =========================================================

gantt_chart(
    best_hybrid_overall["Best_Sequence"],
    best_hybrid_overall["Processing_Times"],
    "results/gantt_chart_best_hybrid.png",
    "Gantt Chart of Best Hybrid GA Schedule"
)

# =========================================================
# FINAL OUTPUT
# =========================================================

print("\n==============================")
print("METHOD COMPARISON SUMMARY")
print("==============================")
display(summary_df)

print("\n==============================")
print("BASELINE GA vs HYBRID GA")
print("==============================")
display(comparison_df)

print("\n==============================")
print("BEST HYBRID SOLUTION")
print("==============================")
print("Benchmark:", best_hybrid_overall["Benchmark"])
print("Best Makespan:", best_hybrid_overall["Best_Makespan"])
print("Best Sequence:", best_hybrid_overall["Best_Sequence"])

print("\nFiles saved:")
print("results/method_comparison_summary.csv")
print("results/all_run_details.csv")
print("results/baseline_vs_hybrid_comparison.csv")
print("results/gantt_chart_best_hybrid.png")